In [11]:
import os
import pickle
from pathlib import Path
import numpy as np
import soundfile as sf
from tqdm import tqdm
import shutil
import csv


In [ ]:

RAW_EXTERNAL_DIR = Path("C:\\Users\\Сичкаренко\\source\\dcunet_dataset")  # исходный датасет

DATA_DIR = Path("C:\\Users\\Сичкаренко\\autodition\\data")
RAW_INTERNAL_DIR = DATA_DIR / "raw" / "dcunet_dataset"   
PREPROCESSED_DIR = DATA_DIR / "preprocessed" / "dcunet_dataset"

TARGET_SR = 16000
SR_FOLDER = f"sr_{TARGET_SR//1000}k"

RAW_INTERNAL_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
def load_audio(path):
    audio, sr = sf.read(path)
    return audio.astype(np.float32), sr

# --- Cell 4: TO MONO ---
def to_mono(audio):
    if audio.ndim == 1:
        return audio
    return np.mean(audio, axis=1)

In [15]:

base_dir = RAW_EXTERNAL_DIR / SR_FOLDER
mix_dir = base_dir / "mix_clean"

assert mix_dir.exists(), f"mix_clean not found: {mix_dir}"

filenames = sorted(os.listdir(mix_dir))

metadata = []

for idx, fname in enumerate(tqdm(filenames)):
    if not fname.endswith(".wav"):
        continue

    mix_id = f"mix_{idx:04d}"
    out_dir = RAW_INTERNAL_DIR / mix_id
    sources_dir = out_dir / "sources"

    out_dir.mkdir(parents=True, exist_ok=True)
    sources_dir.mkdir(parents=True, exist_ok=True)

    # paths
    mix_path = base_dir / "mix_clean" / fname
    s1_path = base_dir / "s1" / fname
    s2_path = base_dir / "s2" / fname
    s3_path = base_dir / "s3" / fname
    s4_path = base_dir / "s4" / fname

    # copy files (без декодирования)
    shutil.copy(mix_path, out_dir / "mixture.wav")
    shutil.copy(s1_path, sources_dir / "s1.wav")
    shutil.copy(s2_path, sources_dir / "s2.wav")
    shutil.copy(s3_path, sources_dir / "s3.wav")
    shutil.copy(s4_path, sources_dir / "s4.wav")

    metadata.append([mix_id, fname])

# --- save csv ---
with open(RAW_INTERNAL_DIR / "metadata.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["mix_id", "original_file"])
    writer.writerows(metadata)

print(f"Formatted dataset created: {len(metadata)} samples")


AssertionError: mix_clean not found: C:\Users\Сичкаренко\source\sr_16k\mix_clean

In [8]:
base_dir = RAW_DIR / SR_FOLDER

features = {}
targets = {}
keys = []

filenames = sorted(os.listdir(base_dir / "mix_clean"))

for fname in tqdm(filenames):
    if not fname.endswith(".wav"):
        continue

    key = fname.replace(".wav", "")

    mixture, sources, sr = load_sample(base_dir, fname)

    features[key] = mixture
    targets[key] = sources

    keys.append(key)

print(f"Processed {len(keys)} samples at {TARGET_SR} Hz")

100%|██████████| 512/512 [01:51<00:00,  4.60it/s]

Processed 512 samples at 16000 Hz


In [9]:
np.random.seed(42)
keys = np.array(keys)
np.random.shuffle(keys)

n = len(keys)
train_split = int(0.8 * n)
val_split = int(0.9 * n)

train_keys = keys[:train_split].tolist()
val_keys = keys[train_split:val_split].tolist()
test_keys = keys[val_split:].tolist()

print(len(train_keys), len(val_keys), len(test_keys))

# --- Cell 8: Save ---
with open(PREPROCESSED_DIR / "features.pkl", "wb") as f:
    pickle.dump(features, f)

with open(PREPROCESSED_DIR / "targets.pkl", "wb") as f:
    pickle.dump(targets, f)

with open(PREPROCESSED_DIR / "train_keys.pkl", "wb") as f:
    pickle.dump(train_keys, f)

with open(PREPROCESSED_DIR / "val_keys.pkl", "wb") as f:
    pickle.dump(val_keys, f)

with open(PREPROCESSED_DIR / "test_keys.pkl", "wb") as f:
    pickle.dump(test_keys, f)

print("Saved!")


409 51 52
Saved!


In [10]:
# --- Cell 9: Sanity check ---
k = train_keys[0]
print("Feature shape:", features[k].shape)
print("Target shape:", targets[k].shape)  # (4, T)

Feature shape: (64000,)
Target shape: (4, 64000)
